In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

data = pd.read_csv('../data/contratos_ceara.csv')

print("numero de linhas e colunas ")
data.shape

In [ ]:
data.isnull().sum() 

In [ ]:
## Ajustando colunas 

data["codigoPaisFornecedor"] = "BRA"

data = data.drop(columns=["urlCipi"], errors="ignore")
data = data.drop(columns=["identificadorCipi"], errors="ignore")
data = data.drop(columns=["orgaoSubRogado"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada"], errors="ignore")
data = data.drop(columns=["orgaoSubRogado.cnpj"], errors="ignore")
data = data.drop(columns=["orgaoSubRogado.razaoSocial"], errors="ignore")
data = data.drop(columns=["orgaoSubRogado.esferaId"], errors="ignore")
data = data.drop(columns=["orgaoSubRogado.poderId"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.codigoUnidade"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.ufSigla"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.municipioNome"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.nomeUnidade"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.codigoIbge"], errors="ignore")
data = data.drop(columns=["unidadeSubRogada.ufNome"], errors="ignore")
data = data.drop(columns=["niFornecedorSubContratado"], errors="ignore")
data = data.drop(columns=["nomeFornecedorSubContratado"], errors="ignore")
data = data.drop(columns=["tipoPessoaSubContratada"], errors="ignore")
data = data.drop(columns=["informacaoComplementar"], errors="ignore")


# Deixar em ordem alfabetica pelo nome do municipio

data = data.sort_values(by="unidadeOrgao.municipioNome", ascending=True)

# Deixa essa coluna numerica
data['valorGlobal'] = pd.to_numeric(data['valorGlobal'], errors='coerce')




In [ ]:
data.to_csv('../data/contratos_ceara_limpo.csv', index=False, encoding='utf-8')

In [ ]:
tops = data['unidadeOrgao.municipioNome'].value_counts()

print("Quantidade de contratos feitas por cada municipio")
print(tops)

plt.figure(figsize=(20, 6))
tops.plot(kind='bar')

plt.xlabel('Município')
plt.ylabel('Quantidade de Contratos')

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
stats = data.groupby("unidadeOrgao.municipioNome")["valorGlobal"].describe()

In [ ]:

plt.figure(figsize=(12, 35))


desc_media_valorGlobal = stats.sort_values(by="mean", ascending=False).head(187).index
data_filtrada = data[data["unidadeOrgao.municipioNome"].isin(desc_media_valorGlobal)]


sns.boxplot(x="valorGlobal", y="unidadeOrgao.municipioNome", data=data_filtrada, palette="viridis")


plt.title("Distribuição de Valor Global por Município ", fontsize=15)
plt.xlabel("Valor Global (R$)", fontsize=12)
plt.ylabel("Município", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)

plt.show()

In [ ]:
maior_por_municipio = data.groupby("unidadeOrgao.municipioNome")["valorGlobal"].max().sort_values(ascending=False)

print(maior_por_municipio)


In [ ]:
import plotly.graph_objects as go

# 1. Ordenando (do maior para o menor)
stats_sorted = stats.sort_values(by="mean", ascending=True) # Ascending True para o maior ficar no topo no gráfico horizontal

# 2. Criando a figura
fig = go.Figure()

fig.add_trace(go.Bar(
    y=stats_sorted.index,  # Municípios no eixo Y agora
    x=stats_sorted['mean'], # Valores no eixo X
    orientation='h',        # Transforma em barras horizontais
    marker_color='royalblue',
    customdata=stats_sorted[['min', 'std', '25%', '50%', '75%', 'max']],
    hovertemplate="<br>".join([
        "<b>Município: %{y}</b>",
        "Média: R$ %{x:,.2f}",
        "Desvio Padrão (std): R$ %{customdata[1]:,.2f}", 
        "Mínimo: R$ %{customdata[0]:,.2f}",             
        "Máximo: R$ %{customdata[5]:,.2f}",            
        "<extra></extra>" 
    ])
))

# 3. Ajustando o Layout para acomodar 182 cidades
fig.update_layout(
    title='Estatísticas por Município (Arraste para ver todos)',
    xaxis_title='Média de Valor Global (R$)',
    yaxis_title='Municípios',
    # Aumentamos a altura para que cada cidade tenha seu espaço
    height=3000, 
    margin=dict(l=200), # Margem maior na esquerda para os nomes das cidades
    template="plotly_white"
)

fig.show()

In [ ]:
# Filtra o dataframe original e aplica o describe
describe_senador = data[data["unidadeOrgao.municipioNome"] == "Senador Pompeu"]["valorGlobal"].describe()

print(describe_senador)